# 03 — Physics-Informed Microgrid Energy Simulation
Simulates:
1. **Solar PV Generation** (Temperature derating + Polar Night shutoff)
2. **Wind Turbine Generation** (Piecewise cubic power curve, cut-in, rated, cut-out)
3. **Station Load Profile** (Base load, temperature-dependent heating demand, research load)
4. **Lithium-Ion Battery Storage (BESS)** (Charge/discharge efficiency, sub-zero cold derating)
5. **Diesel Generator Sets** (Fuel consumption curves, startup costs, minimum loading)


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))


## 1. Wind Turbine Power Curve Simulation


In [ ]:
from src.simulation.wind import simulate_wind_power
speeds = np.linspace(0, 30, 300)
power_curve = [simulate_wind_power(s, capacity_kw=100.0, cut_in_speed=3.0, rated_speed=12.0, cut_out_speed=25.0) for s in speeds]

plt.figure(figsize=(9, 4.5))
plt.plot(speeds, power_curve, color='dodgerblue', lw=2.5)
plt.axvline(3.0, color='gray', linestyle=':', label='Cut-in (3 m/s)')
plt.axvline(12.0, color='green', linestyle=':', label='Rated (12 m/s)')
plt.axvline(25.0, color='red', linestyle=':', label='Cut-out (25 m/s)')
plt.title("Polar Wind Turbine Power Curve (100 kW Rated)")
plt.xlabel("Wind Speed (m/s)")
plt.ylabel("Power Output (kW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 2. Solar PV Output with Thermal Derating


In [ ]:
from src.simulation.solar import simulate_solar_power
irradiance = np.linspace(0, 1000, 200)
p_cold = [simulate_solar_power(g, ambient_temp_c=-20.0, capacity_kw=50.0) for g in irradiance]
p_warm = [simulate_solar_power(g, ambient_temp_c=25.0, capacity_kw=50.0) for g in irradiance]

plt.figure(figsize=(9, 4.5))
plt.plot(irradiance, p_cold, label='T = -20°C (High Efficiency)', color='deepskyblue', lw=2)
plt.plot(irradiance, p_warm, label='T = +25°C (STC Reference)', color='darkorange', lw=2)
plt.title("Solar PV Power Output vs Irradiance")
plt.xlabel("Solar Irradiance (W/m²)")
plt.ylabel("Power Output (kW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Battery Storage & Cold Weather Derating


In [ ]:
from src.simulation.battery import BatterySimulation
temps = np.linspace(-50, 10, 100)
battery = BatterySimulation(capacity_kwh=500.0, cold_threshold_c=-20.0, cold_derating_factor=0.005)
effective_caps = [battery.get_effective_capacity(t) for t in temps]

plt.figure(figsize=(9, 4))
plt.plot(temps, effective_caps, color='purple', lw=2.5)
plt.axvline(-20.0, color='crimson', linestyle='--', label='Cold Derating Threshold (-20°C)')
plt.title("500 kWh Battery Bank: Effective Capacity vs Ambient Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("Available Capacity (kWh)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
